## Pięć zasad efektywnej inżynierii promptów
1. Jasne instrukcje poprawiają trafność.
2. Przykłady stabilizują odpowiedź.
3. Zdefiniowany format odpowiedzi = przewidywalny wynik.
4. Dzielenie na kroki = lepsze i pełniejsze rozwiązania.
5. Testy i weryfikacja = bezpieczeństwo i poprawność.

In [1]:
# Zainstaluj zależności używane w tym notebooku.
%pip install -qU langchain-core langchain-openai langchain-classic pydantic python-dotenv
# langchain-classic udostępnia ewaluator używany w zasadzie 5.

# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notebookowi na dostęp do sekretu.
# Odkomentuj poniższe linie, aby wczytać sekret bez umieszczania klucza w notebooku:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Lokalnie: umieść OPENAI_API_KEY=your_api_key w pliku .env w katalogu roboczym; nie dodawaj go do repozytorium.

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import bibliotek
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

# Model bazowy
llm = ChatOpenAI(model="gpt-6-luna")

### Zasada 1 - Jasne instrukcje

In [3]:
# Zły prompt — niejasny, bez roli i oczekiwań
bad_prompt = "Napisz funkcję w Pythonie."
print("=== Zły prompt ===")
print(llm.invoke(bad_prompt).content)

# Dobry prompt — jasno określona rola i oczekiwania
good_prompt = """Jesteś ekspertem programującym w Pythonie.
Napisz funkcję w Pythonie, która przyjmuje listę liczb całkowitych
i zwraca nową listę zawierającą tylko liczby parzyste.
Dodaj test jednostkowy w pytest."""
print("\n=== Dobry prompt ===")
print(llm.invoke(good_prompt).content)

=== Zły prompt ===
Co ma robić funkcja? Opisz proszę, jakie dane ma przyjmować i jaki wynik zwracać.

=== Dobry prompt ===
```python
def tylko_parzyste(liczby: list[int]) -> list[int]:
    """Zwraca nową listę zawierającą tylko liczby parzyste."""
    return [liczba for liczba in liczby if liczba % 2 == 0]
```

Test w `test_parzyste.py`:

```python
from twoj_modul import tylko_parzyste


def test_tylko_parzyste():
    assert tylko_parzyste([1, 2, 3, 4, -6, 0]) == [2, 4, -6, 0]
    assert tylko_parzyste([1, 3, 5]) == []
    assert tylko_parzyste([]) == []
```

Uruchom testy poleceniem `pytest`. Zastąp `twoj_modul` nazwą pliku, w którym znajduje się funkcja.


### Zasada 2 - Używaj przykładów

In [4]:
# Zadanie: wygeneruj tagi na podstawie treści strony firmy

# Zero-shot
zero_shot = """Podaj tagi opisujące firmę na podstawie tekstu strony:
Firma Lego produkuje zabawki dla dzieci."""
print("=== Zero-shot ===")
print(llm.invoke(zero_shot).content)

# One-shot
one_shot = """Podaj maksymalnie trzy tagi opisujące firmę na podstawie tekstu strony.
Przykład:
Tekst: Firma Lego produkuje klocki dla dzieci.
Tagi: zabawki, klocki, dzieci

Teraz:
Tekst: Firma Nike produkuje odzież i buty sportowe.
Tagi:"""
print("\n=== One-shot ===")
print(llm.invoke(one_shot).content)

# Few-shot
few_shot = """Podaj maksymalnie trzy tagi opisujące firmę na podstawie tekstu strony.

Przykład 1:
Tekst: Firma Lego produkuje klocki dla dzieci.
Tagi: zabawki, klocki, dzieci

Przykład 2:
Tekst: Firma Nike produkuje odzież i buty sportowe.
Tagi: sport, odzież, obuwie

Teraz:
Tekst: Firma Tesla produkuje samochody elektryczne i magazyny energii.
Tagi:"""
print("\n=== Few-shot ===")
print(llm.invoke(few_shot).content)


=== Zero-shot ===
LEGO, producent zabawek, zabawki dla dzieci, branża zabawkarska, produkcja zabawek

=== One-shot ===
odzież sportowa, buty sportowe, sport

=== Few-shot ===
samochody elektryczne, magazyny energii, motoryzacja


In [5]:
prompt = """Stwórz listę 5 sloganów reklamowych dla kawiarni.
Zastosuj styl z przykładowych sloganów załączonych poniżej:
"U nas dostaniesz kawę na ławę!"
"Kawa szybko tanio i konkretnie!"
"""
print(llm.invoke(prompt).content)

1. „Wpadnij na kawę — dzień od razu ciekawszy!”
2. „Mała kawa, wielka sprawa!”
3. „Dobra kawa? U nas to podstawa!”
4. „Kawa, ciastko i pośpiech znika!”
5. „Łyk kawy i świat nabiera smaku!”


In [6]:
prompt = """Stwórz listę 5 sloganów reklamowych dla kawiarni.
Zastosuj styl z przykładowych sloganów załączonych poniżej:
"Nasza kawa jest niczym spacer w rajskim ogrodzie w słoneczny radosny dzień."
"Pijąc naszą kawę poczujesz się tak, jakby anioł zatańczył Tobie na języku."
"""
print(llm.invoke(prompt).content)

1. „Nasza kawa otuli Cię ciepłem jak kocyk utkany z porannych promieni słońca.”
2. „Każdy łyk to mała podróż do krainy, gdzie pachnie czekoladą, a czas płynie wolniej.”
3. „Nasza kawa smakuje jak uśmiech słońca, który wpadł na chwilę do Twojej filiżanki.”
4. „Poczuj, jak aromat kawy unosi Cię lekko niczym balon ponad codziennymi troskami.”
5. „W naszej kawiarni każda filiżanka skrywa odrobinę magii i smak przytulnego popołudnia.”


### Zasada 3 - Zdefiniuj format odpowiedzi
walidacja wejścia i wyjścia z wykorzystaniem biblioteki Pydantic

In [7]:
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Definicja schematu wyjścia (Pydantic)
class CityGuide(BaseModel):
    city: str = Field(..., description="Miasto, którego dotyczy przewodnik")
    summary: str = Field(..., description="Krótki opis miasta (2–3 zdania)")
    must_do: List[str] = Field(..., description="Lista 3–5 rzeczy do zrobienia")

# LLM + structured output
llm = ChatOpenAI(model="gpt-6-luna")
structured_llm = llm.with_structured_output(CityGuide)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Jesteś ekspertem od podróży. Odpowiadaj po polsku, zwięźle."),
    ("user", "Stwórz krótki przewodnik po {city} dla {days}-dniowej wizyty.")
])

chain = prompt | structured_llm

class GuideRequest(BaseModel):
    city: str = Field(min_length=2)
    days: int = Field(ge=1, le=7)

req = GuideRequest(city="Wrocław", days=2)

result: CityGuide = chain.invoke(req.model_dump())
print(result)
print(result.model_dump_json(indent=2))

city='Wrocław' summary='Wrocław zachwyca kolorowym Rynkiem, zabytkowymi mostami i klimatycznymi wyspami nad Odrą. Dwa dni wystarczą, by poznać najważniejsze atrakcje i spróbować lokalnych przysmaków.' must_do=['Dzień 1: odwiedź Rynek, zobacz ratusz i zajrzyj na Ostrów Tumski, najlepiej wieczorem.', 'Dzień 1: przejdź się nad Odrą i zobacz panoramę miasta z Mostu Pokutnic.', 'Dzień 2: zwiedź Panoramę Racławicką oraz okolice Hali Stulecia i Pergoli.', 'Dzień 2: poszukaj wrocławskich krasnali i odpocznij w jednej z kawiarni przy Rynku.']
{
  "city": "Wrocław",
  "summary": "Wrocław zachwyca kolorowym Rynkiem, zabytkowymi mostami i klimatycznymi wyspami nad Odrą. Dwa dni wystarczą, by poznać najważniejsze atrakcje i spróbować lokalnych przysmaków.",
  "must_do": [
    "Dzień 1: odwiedź Rynek, zobacz ratusz i zajrzyj na Ostrów Tumski, najlepiej wieczorem.",
    "Dzień 1: przejdź się nad Odrą i zobacz panoramę miasta z Mostu Pokutnic.",
    "Dzień 2: zwiedź Panoramę Racławicką oraz okolice Ha

### Zasada 4 - Dziel złożone zadania na kroki

#### Przygotowanie planu wycieczki

In [8]:
# Zły prompt — wszystko naraz
bad_prompt = """Przygotuj trzydniowy plan zwiedzania Poznania z budżetem 300 euro,
uwzględniając atrakcje, restauracje, transport i mapy."""
print("=== Zły prompt ===")
print(llm.invoke(bad_prompt).content[:600], "...")

# Dobry prompt — krok po kroku
good_step1 = "Wypisz najważniejsze atrakcje kulturalne w Poznaniu z godzinami otwarcia."
step1 = llm.invoke(good_step1).content
print("\n=== Dobry prompt — krok 1 ===")
print(step1[:600], "...")

good_step2 = f"Na podstawie tej listy ułóż plan zwiedzania na 3 dni, maks 4 atrakcje dziennie. Atrakcje: {step1}"
step2 = llm.invoke(good_step2).content
print("\n=== Dobry prompt — krok 2 ===")
print(step2[:600], "...")


=== Zły prompt ===
## Poznań w 3 dni — budżet do 300 €

**Założenie:** podróżujesz solo, a budżet obejmuje 2 noclegi, jedzenie, lokalny transport i atrakcje. Nie obejmuje dojazdu do Poznania ani zakupów. Przy kursie orientacyjnym **1 € ≈ 4,30 zł**, 300 € to około **1290 zł**.

### Dzień 1: Stary Rynek i centrum

- **Rano:** śniadanie w piekarni lub hostelu, potem spacer po Starym Rynku. W południe warto zobaczyć koziołki na ratuszu — sprawdź wcześniej godziny pokazu.
- **Atrakcja:** Rogalowe Muzeum Poznania. Pokaz trwa około godziny; warto zarezerwować miejsce i sprawdzić aktualną cenę oraz godziny.
- **Po połud ...

=== Dobry prompt — krok 1 ===
Poniżej najważniejsze miejsca związane z historią i kulturą Poznania. **Godziny są orientacyjne** — mogą się zmieniać w zależności od sezonu, świąt i wystaw, więc przed wizytą warto sprawdzić stronę danej instytucji.

| Miejsce | Typowe godziny otwarcia |
|---|---|
| **Muzeum Narodowe w Poznaniu** | Zwykle wt.–niedz. 10:00–17:00; poniedziałek 

#### Generowanie artykułu w krokach

In [9]:
ARTICLE_SUBJECT = "Jakie kompetencje będą najcenniejsze u programistów za 10 lat."

plan = f"Przygotuj plan składający sie z 3 punktów artykułu na temat: {ARTICLE_SUBJECT}."
planGenerated = llm.invoke(plan).content
print(planGenerated, "...")

### Plan artykułu: „Jakie kompetencje będą najcenniejsze u programistów za 10 lat?”

1. **Współpraca ze sztuczną inteligencją** – wykorzystywanie narzędzi AI do tworzenia i testowania kodu, a także umiejętność weryfikowania ich wyników i rozumienia ograniczeń.

2. **Solidne podstawy techniczne i gotowość do nauki** – znajomość algorytmów, architektury systemów i zasad bezpieczeństwa, połączona ze zdolnością szybkiego przyswajania nowych technologii.

3. **Kompetencje ludzkie i odpowiedzialność** – komunikacja, współpraca, rozwiązywanie problemów oraz uwzględnianie etycznych i społecznych skutków tworzonych rozwiązań. ...


In [10]:
MAX_WORDS_IN_PARAGRAPH = 200

prevParagraphContent = ""

for i in range(1,4):
    paragraphPrompt = f"Przygotuj tylko treść {i}. akapitu artykułu: {ARTICLE_SUBJECT}. Maksymalna długość tekstu to {MAX_WORDS_IN_PARAGRAPH}. Wygeneruj tylko tekst jednego paragrafu. Plan całego artykułu:\n {planGenerated}. Treść poprzednich akapitów: {prevParagraphContent}."
    paragraphContent = llm.invoke(paragraphPrompt).content
    prevParagraphContent += paragraphContent + "\n"
    print(f"Akapit {i}.", paragraphContent, "\n\n")

Akapit 1. Za dziesięć lat programowanie będzie wymagało czegoś więcej niż sprawnego pisania kodu. Najcenniejsi okażą się specjaliści, którzy połączą wiedzę techniczną z umiejętnością współpracy z AI i ludźmi. 


Akapit 2. Niezmiennie ważne będą solidne podstawy: znajomość algorytmów, architektury systemów i bezpieczeństwa. Przewagę zyska też gotowość do szybkiego uczenia się, gdy technologie i narzędzia będą się zmieniać. 


Akapit 3. Równie ważne będą kompetencje ludzkie: jasna komunikacja, współpraca i skuteczne rozwiązywanie problemów. Programiści będą też musieli brać odpowiedzialność za etyczne i społeczne skutki tworzonych rozwiązań. 




### Zasada 5 - Testuj i weryfikuj wyniki

In [11]:
!pip install -q langchain langchain-openai langchain-community python-dotenv

In [12]:
from langchain_classic.evaluation import load_evaluator
from dotenv import load_dotenv

load_dotenv()

evaluator = load_evaluator("embedding_distance", embeddings_model="openai")

result = evaluator.evaluate_strings(
    prediction="Stolica Polski to Warszawa",
    reference="coś zupełnie innego"
)

print(result)


{'score': 0.2288585785726368}
